# 1. Setup & Clone Repository
Clone the GitHub repository and install dependencies.

In [ ]:
!rm -rf /content/Malaria-Parasite-Detector
%cd /content

/content


In [ ]:
!git clone https://github.com/10Unknownboy/Malaria-Parasite-Detector.git
%cd Malaria-Parasite-Detector
!pip install -r requirements_colab.txt

Cloning into 'Malaria-Parasite-Detector'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (101/101), done.
Receiving objects: 100% (139/139), 103.62 KiB | 964.00 KiB/s, done.
remote: Total 139 (delta 74), reused 101 (delta 36), pack-reused 0 (from 0)
Resolving deltas: 100% (74/74), done.
/content/Malaria-Parasite-Detector


# 2. Download Dataset via Kagglehub
Download the dataset without an API key and move it to the data directory.

In [ ]:
import kagglehub
import os
import shutil

# Download latest version
path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print("Path to dataset files:", path)

# The downloaded path will contain a 'cell_images' folder, which contains 'Parasitized' and 'Uninfected'
source_dir = os.path.join(path, "cell_images")
if not os.path.exists(os.path.join(source_dir, "Parasitized")):
    # Sometimes kagglehub extracts it differently, check subfolders
    for root, dirs, files in os.walk(path):
        if "Parasitized" in dirs and "Uninfected" in dirs:
            source_dir = root
            break

# Create data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Copy the folders
print("Copying Parasitized images...")
!cp -r "$source_dir/Parasitized" data/
print("Copying Uninfected images...")
!cp -r "$source_dir/Uninfected" data/
print("Dataset ready at data/")

Using Colab cache for faster access to the 'cell-images-for-detecting-malaria' dataset.
Path to dataset files: /kaggle/input/cell-images-for-detecting-malaria
Copying Parasitized images...
Copying Uninfected images...
Dataset ready at data/


# 3. Execute Training Pipeline
Run the local training orchestrator to train all models, evaluate them, and generate Grad-CAMs.

In [ ]:
!python main.py

 Starting Local Training Pipeline

 Device: cuda

--- 1. Sanity Checks ---

    Sanity Check Report
    Dataset Size
       Found 27,558 images (expected ~27,558, tolerance ±5 %: [26,180 – 28,935])
    Class Balance
       Parasitized: 13,779 (50.0%)  |  Uninfected: 13,779 (50.0%)  |  Acceptable: 45%–55%
    Image Loading
       Successfully loaded 50 random sample images.
    Model Output Shape
       No models provided — skipped.
    All checks passed — ready for training!


--- 2. Data Preparation ---
   Data insights saved to: /content/Malaria-Parasite-Detector/results/data_insights.txt

    Dataset Class Balance
  Total images       : 27,558
  Parasitized  (1)   : 13,779  (50.0%)
  Uninfected   (0)   : 13,779  (50.0%)


--- 3. Initialize Models ---
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 188MB/s]
Downloading: "https://download.pytorch.org/models/mobilenet_v2-

# 4. Package and Download Results
Move all charts, graphs, and metrics into the `models/` folder, zip it, and download.

In [ ]:
import os
from google.colab import files

# Move results and reports into the models folder so they get zipped together
if os.path.exists("results"):
    !cp -r results models/
if os.path.exists("reports"):
    !cp -r reports models/

print("Compressing models directory...")
!zip -r /content/malaria_models_and_results.zip models/

print("Downloading zip file...")
files.download('/content/malaria_models_and_results.zip')

Compressing models directory...
  adding: models/ (stored 0%)
  adding: models/results/ (stored 0%)
  adding: models/results/robustness/ (stored 0%)
  adding: models/results/robustness/mobilenetv2_robustness.json (deflated 65%)
  adding: models/results/robustness/mobilenetv2_blur.png (deflated 22%)
  adding: models/results/robustness/mobilenetv2_noise.png (deflated 22%)
  adding: models/results/final_results_summary.json (deflated 58%)
  adding: models/results/sample_predictions/ (stored 0%)
  adding: models/results/sample_predictions/mobilenetv2_samples.png (deflated 9%)
  adding: models/results/sample_predictions/resnet18_samples.png (deflated 8%)
  adding: models/results/sample_predictions/simple_cnn_samples.png (deflated 9%)
  adding: models/results/confusion_matrices/ (stored 0%)
  adding: models/results/confusion_matrices/simple_cnn_cm.png (deflated 16%)
  adding: models/results/confusion_matrices/mobilenetv2_cm.png (deflated 15%)
  adding: models/results/confusion_matrices/resne

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 5. Run Streamlit Dashboard
Use Ngrok to expose the Streamlit app to the public internet.

In [ ]:
import getpass
from pyngrok import ngrok, conf
import time

print("Enter your ngrok authtoken (get it for free at https://dashboard.ngrok.com/get-started/your-authtoken):")
authtoken = getpass.getpass()
conf.get_default().auth_token = authtoken

# Terminate open tunnels if they exist
ngrok.kill()

# Open a tunnel on the default Streamlit port (8501)
public_url = ngrok.connect(8501).public_url
print(f"\n Streamlit Dashboard is starting! It will be live at: {public_url}\n")

# Run Streamlit in the background
!nohup streamlit run app.py > streamlit_logs.txt 2>&1 &
time.sleep(3)
print("Dashboard is running. Check the URL above!")